<a href="https://colab.research.google.com/github/divyam-gawde/grapesjs_demo/blob/main/novelfire_epub_generator_with_updated_progress_bar_and_range_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#@title 📘 Novel Scraper (With Chapter Range + Cover)
link = "https://novelfire.net/book/my-sloppy-husband-is-actually-a-handsome-bigshot"  #@param {type:"string"}
chapter_range = "1 50"  #@param {type:"string"}

import requests
import base64
import time
import random
import threading
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor
from IPython.display import clear_output

# -----------------------------
# CONFIG
# -----------------------------
MAX_WORKERS = 3
BASE_DELAY = 1.5
MAX_RETRIES = 5

# -----------------------------
# Parse range
# -----------------------------
def parse_range(rng):
    try:
        a, b = rng.strip().split()
        return int(a), int(b)
    except:
        return None, None

range_start, range_end = parse_range(chapter_range)

# -----------------------------
# Session
# -----------------------------
def create_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": random.choice([
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "Mozilla/5.0 (X11; Linux x86_64)",
            "Mozilla/5.0 (Macintosh)"
        ]),
        "Referer": "https://novelfire.net/"
    })
    return s

session = create_session()

# -----------------------------
# MAIN PAGE
# -----------------------------
main_page = session.get(link)
soup = BeautifulSoup(main_page.text, "lxml")

# Title & Author
title_tag = soup.find("h1", class_="novel-title")
novel_title = title_tag.get_text(strip=True) if title_tag else "Unknown Title"

author_block = soup.find("div", class_="author")
author_spans = author_block.find_all("span", attrs={"itemprop": "author"}) if author_block else []
authors = [span.get_text(strip=True) for span in author_spans]
novel_author = ", ".join(authors) if authors else "Unknown Author"

# -----------------------------
# ✅ COVER BLOCK (FIXED)
# -----------------------------
cover_data = None
cover_tag = soup.select_one("figure.cover img")

if cover_tag and cover_tag.get("src"):
    src = cover_tag["src"]

    if src.startswith("data:"):
        header, b64data = src.split(",", 1)
        cover_data = base64.b64decode(b64data)
    else:
        try:
            cover_url = urljoin(link, src)
            resp = session.get(cover_url)
            resp.raise_for_status()
            cover_data = resp.content
        except:
            print("⚠️ Cover download failed")

# -----------------------------
# CHAPTER LIST
# -----------------------------
chapter_link_tag = soup.find("a", class_="grdbtn chapter-latest-container")
chapter_list_url = urljoin(link, chapter_link_tag.get("href"))

chapter_list_page = session.get(chapter_list_url)
chapters_soup = BeautifulSoup(chapter_list_page.text, "lxml")

chapters = []

max_page = 1
for a in chapters_soup.select(".pagination a.page-link"):
    txt = a.get_text(strip=True)
    if txt.isdigit():
        max_page = max(max_page, int(txt))

for page_num in range(1, max_page + 1):
    if page_num == 1:
        page_soup = chapters_soup
    else:
        page_url = f"{chapter_list_url}?page={page_num}"
        page_req = session.get(page_url)
        page_soup = BeautifulSoup(page_req.text, "lxml")

    for a in page_soup.select('#chpagedlist ul.chapter-list li a'):
        url = urljoin(chapter_list_url, a.get("href"))

        no_tag = a.select_one(".chapter-no")
        title_tag = a.select_one(".chapter-title")

        chapters.append({
            "no": no_tag.get_text(strip=True) if no_tag else "",
            "title": title_tag.get_text(strip=True) if title_tag else "",
            "url": url,
        })

# -----------------------------
# FILTER RANGE
# -----------------------------
def extract_number(text):
    match = re.search(r'\d+', text)
    return int(match.group()) if match else None

if range_start and range_end:
    chapters = [
        ch for ch in chapters
        if extract_number(ch["no"]) and range_start <= extract_number(ch["no"]) <= range_end
    ]

# -----------------------------
# STATUS
# -----------------------------
progress = {"done": 0, "total": len(chapters), "last": ""}
lock = threading.Lock()

def show_status():
    clear_output(wait=True)
    print(f"📖 {novel_title}")
    print(f"✍️ {novel_author}")
    print(f"📚 Chapters: {progress['total']}")
    print(f"🚀 Progress: {progress['done']} / {progress['total']}")
    print(f"⚡ Last: {progress['last']}")

# -----------------------------
# DOWNLOAD
# -----------------------------
def fetch_chapter(chapter):
    for attempt in range(MAX_RETRIES):
        try:
            s = create_session()
            resp = s.get(chapter["url"], timeout=15)

            if resp.status_code == 429:
                time.sleep((2 ** attempt) + random.uniform(1, 3))
                continue

            soup = BeautifulSoup(resp.text, "lxml")
            container = soup.find("div", id="chapter-container")
            content_div = container.find("div", id="content") if container else None

            chapter["content"] = str(content_div) if content_div else "<p>No content</p>"

            with lock:
                progress["done"] += 1
                progress["last"] = chapter["title"]
                show_status()

            time.sleep(BASE_DELAY + random.uniform(0.5, 1.5))
            return chapter

        except:
            time.sleep((2 ** attempt) + random.uniform(1, 2))

    chapter["content"] = "<p>Failed</p>"
    return chapter

# -----------------------------
# RUN
# -----------------------------
show_status()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    list(executor.map(fetch_chapter, chapters))

print("\n🎉 Completed!")

📖 My Sloppy Husband Is Actually a Handsome Bigshot!
✍️ Nalan Xueyang
📚 Chapters: 50
🚀 Progress: 50 / 50
⚡ Last: Chapter 50 I Truly Love Him

🎉 Completed!


In [ ]:
#@title 1 Test Generate Epub
!pip install -q ebooklib
from ebooklib import epub
import requests

# 1. Create EPUB book object
book = epub.EpubBook()

book.set_identifier(novel_title)
book.set_title(novel_title)
book.set_language("en")
book.add_author(novel_author)

# 2. Add cover if we have it
if cover_data:
    book.set_cover("cover.jpg", cover_data)
    print("Cover added to EPUB.")
else:
    print("No cover added (no cover_data).")

# 3. Create chapters and add to the book
epub_chapters = []

for chapter in chapters:
    chapter_html = chapter["content"]
    file_name = f"chapter_{chapter['no']}.xhtml"

    c = epub.EpubHtml(
        title=chapter["title"],
        file_name=file_name,
        lang="en"
    )

    c.content = f"""
    <html xmlns="http://www.w3.org/1999/xhtml">
        <head>
            <title>{chapter['title']}</title>
        </head>
        <body>
            <h2>{chapter['title']}</h2>
            {chapter_html}
        </body>
    </html>
    """

    book.add_item(c)
    epub_chapters.append(c)

# 4. TOC & spine
book.toc = tuple(epub_chapters)
book.spine = ['nav'] + epub_chapters

# 5. Navigation files
book.add_item(epub.EpubNcx())
book.add_item(epub.EpubNav())

# 6. Write EPUB file
safe_title = novel_title.replace(" ", "_")
epub_file_name = f"{safe_title}.epub"
epub.write_epub(epub_file_name, book, {})

print("✔ EPUB successfully created:", epub_file_name)

# -------------------------------------------------------
# 7. Upload EPUB to GoFile
# -------------------------------------------------------

upload_url = "https://upload.gofile.io/uploadfile"

with open(epub_file_name, "rb") as f:
    files = {"file": f}
    response = requests.post(upload_url, files=files)

data = response.json()

if data.get("status") == "ok":
    download_link = data["data"].get("downloadPage")
    direct_link = data["data"].get("directLink")

    print("📤 Uploaded to GoFile!")

    if download_link:
        print("Download page:", download_link)

    if direct_link:
        print("Direct file:", direct_link)

else:
    print("Upload failed:", data)

Cover added to EPUB.
✔ EPUB successfully created: My_Sloppy_Husband_Is_Actually_a_Handsome_Bigshot!.epub
📤 Uploaded to GoFile!
Download page: https://gofile.io/d/AeY5Uy
